##Subset Analysis (post result diagnostic)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split, Dataset, Subset
import timm
import numpy as np
import os
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch: 2.11.0+cu128
GPU: NVIDIA L4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import shutil

with zipfile.ZipFile('/content/drive/MyDrive/data_combined_8k.zip', 'r') as z:
    for member in z.namelist():
        fixed_path = member.replace('\\', '/')
        target = os.path.join('/content', fixed_path)
        if member.endswith('\\') or '.' not in os.path.basename(fixed_path):
            os.makedirs(target, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(member) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)

print("Real frames:", len(os.listdir('/content/data_combined_8k/real')))
print("Fake frames:", len(os.listdir('/content/data_combined_8k/fake')))

Mounted at /content/drive
Real frames: 8000
Fake frames: 8000


In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        image = self.subset.dataset.loader(path)
        return self.transform(image), label

full_dataset = ImageFolder('/content/data_combined_8k', transform=None)
print("Classes:", full_dataset.classes)

train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

split_generator = torch.Generator().manual_seed(42)

train_sub, val_sub, test_sub = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=split_generator
)


test_set = TransformSubset(test_sub, test_transform)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)

print(f"Test set size: {len(test_set)}")

Classes: ['fake', 'real']
Test set size: 2400


In [ ]:
class FrequencyBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )
    def forward(self, x):
        gray = x.mean(dim=1, keepdim=True)
        fft = torch.fft.fft2(gray)
        fft = torch.fft.fftshift(fft, dim=(-2, -1))
        magnitude = torch.abs(fft)
        magnitude = torch.log1p(magnitude)
        magnitude = (magnitude - magnitude.mean(dim=(-2, -1), keepdim=True)) / \
                    (magnitude.std(dim=(-2, -1), keepdim=True) + 1e-6)
        out = self.features(magnitude)
        out = self.classifier(out)
        return out

class CombinedDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.xception = timm.create_model('xception', pretrained=True, num_classes=0)
        xception_features = self.xception.num_features
        self.freq_branch = FrequencyBranch()
        freq_features = 128
        self.classifier = nn.Sequential(
            nn.Linear(xception_features + freq_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        spatial_features = self.xception(x)
        freq_features = self.freq_branch(x)
        combined = torch.cat([spatial_features, freq_features], dim=1)
        out = self.classifier(combined)
        return out


class FrequencyBranchRefined(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64*4*4, 128), nn.ReLU(inplace=True), nn.Dropout(0.3))
    def forward(self, x):
        fft = torch.fft.fft2(x)
        fft = torch.fft.fftshift(fft, dim=(-2, -1))
        magnitude = torch.abs(fft)
        magnitude = torch.log1p(magnitude)
        magnitude = (magnitude - magnitude.mean(dim=(-2,-1), keepdim=True)) / (magnitude.std(dim=(-2,-1), keepdim=True) + 1e-6)
        phase = torch.angle(fft) / torch.pi
        freq_input = torch.cat([magnitude, phase], dim=1)
        out = self.features(freq_input)
        return self.classifier(out)

class CombinedDetectorRefined(nn.Module):
    def __init__(self):
        super().__init__()
        self.xception = timm.create_model('xception', pretrained=True, num_classes=0)
        xception_features = self.xception.num_features
        self.freq_branch = FrequencyBranchRefined()
        self.classifier = nn.Sequential(
            nn.Linear(xception_features + 128, 512), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, 128), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(128, 2)
        )
    def forward(self, x_xception, x_freq):
        spatial_features = self.xception(x_xception)
        freq_features = self.freq_branch(x_freq)
        combined = torch.cat([spatial_features, freq_features], dim=1)
        return self.classifier(combined)


class GatedCombinedDetector(nn.Module):
    def __init__(self):
        super().__init__()

        self.xception = timm.create_model(
            'xception',
            pretrained=True,
            num_classes=0
        )

        xception_features = self.xception.num_features

        self.freq_branch = FrequencyBranchRefined()
        freq_features = 128

        fusion_dim = 256

        self.spatial_proj = nn.Sequential(
            nn.Linear(xception_features, fusion_dim),
            nn.ReLU(inplace=True)
        )

        self.freq_proj = nn.Sequential(
            nn.Linear(freq_features, fusion_dim),
            nn.ReLU(inplace=True)
        )

        self.gate = nn.Sequential(
            nn.Linear(fusion_dim * 2, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x_xception, x_freq):

        spatial_features = self.xception(x_xception)
        freq_features = self.freq_branch(x_freq)

        spatial = self.spatial_proj(spatial_features)
        freq = self.freq_proj(freq_features)

        gate_input = torch.cat([spatial, freq], dim=1)

        weights = torch.softmax(
            self.gate(gate_input),
            dim=1
        )

        spatial_weight = weights[:, 0:1]
        freq_weight = weights[:, 1:2]

        fused = (
            spatial_weight * spatial +
            freq_weight * freq
        )

        return self.classifier(fused)


combined_model = CombinedDetector().cuda()  #all loaded for previous experiments/only gated is used in this final notebook
combined_model.load_state_dict(torch.load('/content/drive/MyDrive/best_model_cosinefix.pth'))
combined_model.eval()
print("Old Detector loaded")

xception_model = timm.create_model('xception', pretrained=True, num_classes=2)
xception_model.load_state_dict(torch.load('/content/drive/MyDrive/best_model_baselinefix.pth'))
xception_model = xception_model.cuda()
xception_model.eval()
print("XceptionOnly loaded")

refined_model = CombinedDetectorRefined().cuda()
refined_model.load_state_dict(torch.load('/content/drive/MyDrive/best_model_combinedwimpfftsplitaug.pth'))
refined_model.eval()
print("CombinedDetector (refined) loaded")


gated_model = GatedCombinedDetector().cuda()
gated_model.load_state_dict(
    torch.load('/content/drive/MyDrive/best_model_combinednewgatedfusion.pth')
)
gated_model.eval()
print("Gated model loaded")

/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
Old Detector loaded
XceptionOnly loaded
CombinedDetector (refined) loaded
Gated model loaded


### Splitting Test Subsets

Test samples are separated by filename

In [ ]:
df_indices = []
fs_indices = []
real_indices = []

for i in range(len(test_sub.indices)):
    path, label = full_dataset.samples[test_sub.indices[i]]
    fname = os.path.basename(path)
    if fname.startswith('df_'):
        df_indices.append(i)
    elif fname.startswith('fs_'):
        fs_indices.append(i)
    else:
        real_indices.append(i)

print(f"Deepfakes in test set: {len(df_indices)}")
print(f"FaceShifter in test set: {len(fs_indices)}")
print(f"Real in test set: {len(real_indices)}")



Deepfakes in test set: 597
FaceShifter in test set: 608
Real in test set: 1195


In [ ]:
def get_gate_stats(model, loader):
    model.eval()

    all_weights = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.cuda()

            spatial_features = model.xception(inputs)
            freq_features = model.freq_branch(inputs)

            spatial = model.spatial_proj(spatial_features)
            freq = model.freq_proj(freq_features)

            gate_input = torch.cat([spatial, freq], dim=1)

            weights = torch.softmax(
                model.gate(gate_input),
                dim=1
            )

            all_weights.append(weights.cpu())

    all_weights = torch.cat(all_weights, dim=0)

    spatial = all_weights[:, 0]
    freq = all_weights[:, 1]

    print(f"Spatial mean: {spatial.mean().item():.4f}")
    print(f"Spatial std:  {spatial.std().item():.4f}")
    print(f"Spatial min:  {spatial.min().item():.4f}")
    print(f"Spatial max:  {spatial.max().item():.4f}")

    print(f"Frequency mean: {freq.mean().item():.4f}")
    print(f"Frequency std:  {freq.std().item():.4f}")
    print(f"Frequency min:  {freq.min().item():.4f}")
    print(f"Frequency max:  {freq.max().item():.4f}")

### Fake Only per-Manipulation Gate Analysis

These figures are referenced in the report

In [ ]:
# Fake only subsets
deepfakes_fake_only = Subset(test_set, df_indices)
faceshifter_fake_only = Subset(test_set, fs_indices)

# Fake only loaders
df_fake_loader = DataLoader(
    deepfakes_fake_only,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

fs_fake_loader = DataLoader(
    faceshifter_fake_only,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [ ]:
print("DeepFakes FAKE ONLY ")
get_gate_stats(gated_model, df_fake_loader)

print("\n FaceShifter FAKE ONLY ")
get_gate_stats(gated_model, fs_fake_loader)

DeepFakes FAKE ONLY 
Spatial mean: 0.9804
Spatial std:  0.0263
Spatial min:  0.8176
Spatial max:  0.9999
Frequency mean: 0.0196
Frequency std:  0.0263
Frequency min:  0.0001
Frequency max:  0.1824

 FaceShifter FAKE ONLY 
Spatial mean: 0.9759
Spatial std:  0.0276
Spatial min:  0.7642
Spatial max:  0.9998
Frequency mean: 0.0241
Frequency std:  0.0276
Frequency min:  0.0002
Frequency max:  0.2358
